# ML-KEM Hardware Acceleration: Multi-Run Benchmark
Runs the full Alice-Bob ML-KEM key exchange protocol multiple times, comparing pure software vs FPGA hardware NTT acceleration.
Each run verifies that the hardware and software produce equivalent shared secrets, and the final cell reports average speedup with standard deviation.

In [2]:
import sys
import os
import time
import hashlib
sys.path.append(os.path.abspath('/home/xilinx/jupyter_notebooks/EE712/Project/ml-kem'))

from mlkem.ml_kem import ML_KEM
from mlkem.parameter_set import ML_KEM_768
import mlkem.auxiliary.ntt as ntt_module
import mlkem.k_pke as k_pke_module

from mlkem.auxiliary.ntt import ntt as sw_ntt, ntt_inv as sw_intt
from mlkem.auxiliary.ntt_hardware import ntt as hw_ntt, ntt_inv as hw_intt

# --- Config ---
NUM_RUNS = 20

print(f"Benchmark configured for {NUM_RUNS} runs.")
print("All modules loaded.")

Benchmark configured for 20 runs.
All modules loaded.


In [3]:
def patch_ntt(ntt_fn, ntt_inv_fn):
    """Correctly patches the NTT functions throughout the ML-KEM library."""
    k_pke_module.ntt = ntt_fn
    k_pke_module.ntt_inv = ntt_inv_fn
    ntt_module.ntt = ntt_fn
    ntt_module.ntt_inv = ntt_inv_fn

def stream_cipher_crypt(key: bytes, message: bytes) -> bytes:
    """SHAKE-256 stream cipher for secure communication demo."""
    key_stream = hashlib.shake_256(key).digest(len(message))
    return bytes([m ^ k for m, k in zip(message, key_stream)])

print("Helpers defined.")

Helpers defined.


### Multi-Run Benchmark Loop
Each iteration runs the full ML-KEM key exchange twice (software then hardware), verifies correctness, and records timing.

In [4]:
ml_kem = ML_KEM(ML_KEM_768, fast=False)
bobs_message = b"Hello Alice! ML-KEM successfully secured this channel with Zynq FPGA Acceleration!"

times_sw = []
times_hw = []
equivalence_results = []

print("=" * 64)
print(f"  ML-KEM NTT Hardware Acceleration Benchmark ({NUM_RUNS} runs)")
print("=" * 64)

for run in range(1, NUM_RUNS + 1):
    print(f"\n--- Run {run}/{NUM_RUNS} ---")

    # ── SOFTWARE RUN ──────────────────────────────────────────────
    patch_ntt(sw_ntt, sw_intt)

    t0 = time.time()
    ek_sw, dk_sw = ml_kem.key_gen()
    k_bob_sw, c_sw = ml_kem.encaps(ek_sw)
    k_alice_sw = ml_kem.decaps(dk_sw, c_sw)
    t_sw = (time.time() - t0) * 1000
    times_sw.append(t_sw)

    assert k_alice_sw == k_bob_sw, f"[Run {run}] SW keys do not match!"

    # ── HARDWARE RUN ──────────────────────────────────────────────
    patch_ntt(hw_ntt, hw_intt)

    t0 = time.time()
    ek_hw, dk_hw = ml_kem.key_gen()
    k_bob_hw, c_hw = ml_kem.encaps(ek_hw)
    k_alice_hw = ml_kem.decaps(dk_hw, c_hw)
    t_hw = (time.time() - t0) * 1000
    times_hw.append(t_hw)

    assert k_alice_hw == k_bob_hw, f"[Run {run}] HW keys do not match!"

    # ── EQUIVALENCE CHECK: HW Bob ↔ SW Alice ─────────────────────
    # Bob uses hardware to encapsulate with Alice's SW-generated public key.
    # Alice uses software to decapsulate. If secrets match, HW is mathematically
    # identical to SW per FIPS-203.
    patch_ntt(hw_ntt, hw_intt)
    k_bob_hybrid, c_hybrid = ml_kem.encaps(ek_sw)

    patch_ntt(sw_ntt, sw_intt)
    k_alice_hybrid = ml_kem.decaps(dk_sw, c_hybrid)

    equivalent = (k_bob_hybrid == k_alice_hybrid)
    equivalence_results.append(equivalent)

    # ── SECURE COMMUNICATION DEMO ─────────────────────────────────
    encrypted = stream_cipher_crypt(k_bob_hw, bobs_message)
    decrypted = stream_cipher_crypt(k_alice_hw, encrypted)
    assert decrypted == bobs_message, f"[Run {run}] Message decryption failed!"

    # ── PER-RUN REPORT ────────────────────────────────────────────
    speedup = t_sw / t_hw if t_hw > 0 else 0
    eq_str = "✅ MATCH" if equivalent else "❌ MISMATCH"
    print(f"  Software :  {t_sw:>8.2f} ms  |  HW Shared Secret: {k_alice_sw.hex()[:16]}...")
    print(f"  Hardware :  {t_hw:>8.2f} ms  |  HW Shared Secret: {k_alice_hw.hex()[:16]}...")
    print(f"  Speedup  :  {speedup:>8.2f}x   |  Equivalence: {eq_str}")
    print(f"  Message  :  '{decrypted.decode()[:40]}...' ✅")

print("\n" + "=" * 64)
print("All runs complete.")

  ML-KEM NTT Hardware Acceleration Benchmark (20 runs)

--- Run 1/20 ---
  Software :   3049.62 ms  |  HW Shared Secret: cf8eaed8dfdc1907...
  Hardware :   2405.07 ms  |  HW Shared Secret: 5f1ed8a06fa1327f...
  Speedup  :      1.27x   |  Equivalence: ✅ MATCH
  Message  :  'Hello Alice! ML-KEM successfully secured...' ✅

--- Run 2/20 ---
  Software :   3061.26 ms  |  HW Shared Secret: 5612819889f961fb...
  Hardware :   2433.47 ms  |  HW Shared Secret: 3e2a1ba39c435335...
  Speedup  :      1.26x   |  Equivalence: ✅ MATCH
  Message  :  'Hello Alice! ML-KEM successfully secured...' ✅

--- Run 3/20 ---
  Software :   3394.00 ms  |  HW Shared Secret: c78c70aae216e837...
  Hardware :   2466.57 ms  |  HW Shared Secret: c6deb9dd2969ffdf...
  Speedup  :      1.38x   |  Equivalence: ✅ MATCH
  Message  :  'Hello Alice! ML-KEM successfully secured...' ✅

--- Run 4/20 ---
  Software :   3418.85 ms  |  HW Shared Secret: 6c287a3b02d8bb82...
  Hardware :   2114.72 ms  |  HW Shared Secret: 4a42e3869654d

### Aggregate Results

In [5]:
import statistics

avg_sw = statistics.mean(times_sw)
avg_hw = statistics.mean(times_hw)
std_sw = statistics.stdev(times_sw) if NUM_RUNS > 1 else 0
std_hw = statistics.stdev(times_hw) if NUM_RUNS > 1 else 0

speedups = [sw / hw for sw, hw in zip(times_sw, times_hw)]
avg_speedup = statistics.mean(speedups)
std_speedup = statistics.stdev(speedups) if NUM_RUNS > 1 else 0

all_equiv = all(equivalence_results)

print("=" * 64)
print("           FINAL BENCHMARK SUMMARY")
print("=" * 64)
print(f"  Runs completed       : {NUM_RUNS}")
print(f"")
print(f"  Avg Software time    : {avg_sw:>8.2f} ms  (± {std_sw:.2f} ms)")
print(f"  Avg Hardware time    : {avg_hw:>8.2f} ms  (± {std_hw:.2f} ms)")
print(f"")
print(f"  Per-run speedups     : {', '.join(f'{s:.2f}x' for s in speedups)}")
print(f"  Average Speedup      : {avg_speedup:>8.2f}x  (± {std_speedup:.2f}x)")
print(f"")
print(f"  HW-SW Equivalence    : {'✅ PASSED all runs' if all_equiv else '❌ FAILED on some runs'}")
print("=" * 64)

if avg_speedup > 1.0:
    print(f"\n✅ FPGA Hardware NTT achieves {avg_speedup:.2f}x average protocol speedup")
    print(f"   and is fully FIPS-203 compliant (mathematically identical to software).")
else:
    print(f"\n⚠️  Protocol speedup < 1.0. Check overhead analysis.")

           FINAL BENCHMARK SUMMARY
  Runs completed       : 20

  Avg Software time    :  3268.51 ms  (± 152.72 ms)
  Avg Hardware time    :  2345.05 ms  (± 135.85 ms)

  Per-run speedups     : 1.27x, 1.26x, 1.38x, 1.62x, 1.27x, 1.38x, 1.40x, 1.58x, 1.27x, 1.37x, 1.58x, 1.29x, 1.28x, 1.40x, 1.61x, 1.28x, 1.39x, 1.36x, 1.40x, 1.61x
  Average Speedup      :     1.40x  (± 0.13x)

  HW-SW Equivalence    : ✅ PASSED all runs

✅ FPGA Hardware NTT achieves 1.40x average protocol speedup
   and is fully FIPS-203 compliant (mathematically identical to software).
